In [8]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from config import CAT_VARS, DATA, NUM_VARS
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    confusion_matrix,
    roc_auc_score,
    roc_curve,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

In [9]:
dataset_path = DATA / 'processed' / 'basic_data_prep'
datasets = {f.stem: pd.read_csv(f) for f in sorted(dataset_path.glob("*.csv"))}
for k, v in datasets.items():
    print(f"{k}: \n{v}\n")


X_test: 
      X01_LIMIT_BAL  X02_SEX  X03_EDUCATION  X04_MARRIAGE  X05_AGE  X06_PAY_0  \
0             50000        1              2             2       46         -1   
1            150000        1              1             1       31         -1   
2             50000        1              2             2       25          0   
3            290000        2              1             2       25          0   
4            500000        2              2             1       27         -2   
...             ...      ...            ...           ...      ...        ...   
5995         150000        2              4             2       27         -2   
5996          50000        1              1             2       24          2   
5997         220000        1              1             2       34          0   
5998         120000        1              1             2       26         -1   
5999         200000        1              3             2       33         -2   

      X07_PAY_2  X

In [10]:

preprocess = ColumnTransformer([
    ('num', StandardScaler(), NUM_VARS),
    ('cat', OneHotEncoder(drop='first', handle_unknown='ignore'), CAT_VARS),
])

model = Pipeline([
    ("prep", preprocess),
    ("clf", LogisticRegression(max_iter=1000, C=np.inf)),
])

model.fit(datasets['X_train'], datasets['y_train'])

/Users/benflint/dev/projects/ML_PD_Model/.venv/lib/python3.13/site-packages/sklearn/utils/validation.py:1365: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)


,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('prep', ...), ('clf', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[int64](2,)","[0,1]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](23,)","['X01_LIMIT_BAL','X02_SEX','X03_EDUCATION',...,'X21_PAY_AMT4', 'X22_PAY_AMT5','X23_PAY_AMT6']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,23
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthro

# Validate results

In [11]:
y_score = model.predict_proba(datasets['X_test'])[:, 1]
y_pred = model.predict(datasets['X_test'],)
y_true = datasets['y_test'].to_numpy().ravel()
print(type(y_true))
print(type(y_score))
print(y_true)
print(y_score)

<class 'numpy.ndarray'>
<class 'numpy.ndarray'>
[0 0 0 ... 0 0 0]
[0.10611204 0.11880812 0.2206938  ... 0.15084857 0.11713119 0.04552326]


In [12]:
auc = roc_auc_score(datasets['y_test'], y_score)
gini = 2 * auc - 1
conf_matrix = confusion_matrix(datasets['y_test'], y_pred).tolist()

metrics = {'auc': auc, 'gini': gini, 'confusion matrix':conf_matrix}

print(metrics)

{'auc': 0.7099583281662152, 'gini': 0.41991665633243036, 'confusion matrix': [[4528, 145], [1003, 324]]}


In [13]:
fpr, tpr, thr = roc_curve(datasets['y_test'], y_score)

fig = go.Figure()
fig.add_trace(go.Scatter(x=[0, 1], y=[0, 1], mode="lines", hoverinfo="skip",
                         line={'color': "#B0B4BA", 'width': 1, 'dash': "dash"},
                         showlegend=False))
fig.add_trace(go.Scatter(x=fpr, y=tpr, mode="lines", customdata=thr,
                         line={'color': "#2B6CB0", 'width': 2}, showlegend=False,
                         hovertemplate="FPR %{x:.3f}<br>TPR %{y:.3f}"
                                       "<br>threshold %{customdata:.3f}<extra></extra>"))
fig.update_layout(
    title=f"ROC — logistic champion, test set (AUC {auc:.3f}, Gini {2*auc-1:.3f})",
    xaxis={"title": "False positive rate", "range": [0, 1], "gridcolor": "#EDEFF2", "zeroline": False},
    yaxis={"title": "True positive rate",  "range": [0, 1], "gridcolor": "#EDEFF2", "zeroline": False,
               "scaleanchor": "x", "scaleratio": 1},
    template="plotly_white", width=560, height=560, margin={"l": 60, "r": 30, "t": 60, "b": 60},
)
fig.show()

**Conclusion:** OK, so Gini is working well for the lowest scores but stops working quite as well around the middle range. Can try to get a better performing model to tell whether this is a limitation of the dataset or the model.